# DATALUS POC Quickstart: Pipeline Generativo SIH-SUS

Este notebook demonstra o framework **DATALUS** para geração de dados tabulares sintéticos.
Utilizamos uma amostra real de dados de saúde brasileiros (SIH-SUS) para demonstrar todo o pipeline:
da ingestão de dados brutos à exportação ONNX para o edge, incluindo auditoria e operações generativas.

O **DATALUS** foi projetado para conjuntos de dados governamentais heterogêneos e de alta dimensão, 
garantindo privacidade (alinhamento com a LGPD) e utilidade por meio de modelos de difusão avançados.

## 1. Configuração do Ambiente
Detectando o ambiente e instalando dependências se necessário.

In [ ]:
import os
import sys
from pathlib import Path

def get_working_dir():
    if 'google.colab' in sys.modules:
        return '/content/datalus_poc'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return '/kaggle/working/datalus_poc'
    return './datalus_poc'

WORKING_DIR = get_working_dir()
os.makedirs(WORKING_DIR, exist_ok=True)
print(f'Working directory: {WORKING_DIR}')

# Instalar dependências se estiver no Colab/Kaggle
if 'google.colab' in sys.modules or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    !git clone https://github.com/emanuellcs/datalus.git || true
    %cd datalus
    !pip install -e '.[dev]' pysus polars matplotlib
    sys.path.append(os.getcwd())


## 2. Ingestão de Dados (SIH-SUS)
Baixamos uma pequena amostra de dados do Sistema de Informações Hospitalares (SIH) para São Paulo (SP) usando a API `pysus`.

In [ ]:
from pysus.api import sih
import polars as pl
import numpy as np

print('Downloading SIH-SUS sample...')
# Buscando amostra de SP, Jan 2024 (pequena fatia para POC)
df_pandas = sih(state='SP', year=2024, month=[1])
df_polars = pl.from_pandas(df_pandas)

# Limpar coluna MORTE (Alvo: 1 para óbito, 0 para sobrevivência)
# No SIH-SUS, MORTE=1 é óbito. Garantimos que seja inteiro.
df_polars = df_polars.with_columns(
    pl.col('MORTE').cast(pl.Int64).fill_null(0)
)

# Remover colunas estritamente vazias ou identificadores para focar na modelagem
cols_to_keep = [c for c in df_polars.columns if not c.startswith(('N_AIH', 'IDENT'))]
df_polars = df_polars.select(cols_to_keep).head(10000) # Pequena amostra para velocidade

raw_path = f'{WORKING_DIR}/raw_sih.parquet'
df_polars.write_parquet(raw_path)
print(f'Data saved to {raw_path} | Shape: {df_polars.shape}')

## 3. Ingestão com a CLI do DATALUS
O comando `ingest` infere o esquema, detecta delimitadores e prepara os dados para modelagem.

In [ ]:
!datalus ingest {WORKING_DIR}/raw_sih.parquet {WORKING_DIR}/processed.parquet --schema-path {WORKING_DIR}/schema_config.json --target-column MORTE

## 4. Treinando o Modelo de Difusão
Treinamos o denoiser MLP residual. Para este POC, usamos um número muito baixo de épocas.

In [ ]:
!datalus train {WORKING_DIR}/schema_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/artifacts --epochs 3 --batch-size 1024 --gpu 0

## 5. Geração Ab-initio (Amostragem)
Gerando registros sintéticos inteiramente novos a partir da distribuição aprendida.

In [ ]:
!datalus sample {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet --n-records 5000

## 6. Aumento de Dados (Augmentation)
Anexando linhas sintéticas a um pequeno conjunto de dados existente.

In [ ]:
import polars as pl
synthetic = pl.read_parquet(f'{WORKING_DIR}/synthetic.parquet')
seed_for_augment = synthetic.sample(n=1000)
seed_for_augment.write_parquet(f'{WORKING_DIR}/seed_for_augment.parquet')

!datalus augment {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/seed_for_augment.parquet {WORKING_DIR}/augmented.parquet --n-records 1000

## 7. Balanceamento de Classe Minoritária
Gerando registros para balancear classes de destino específicas (ex: mortalidade).

In [ ]:
!datalus balance {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/balanced.parquet MORTE '{"1": 2000}'

## 8. Inpainting Tabular (RePaint)
Preenchendo valores faltantes (nulos) preservando os campos observados.

In [ ]:
import polars as pl
processed = pl.read_parquet(f'{WORKING_DIR}/processed.parquet')
# Criar valores ausentes na coluna 'IDADE' para demonstração
incomplete = processed.head(100).with_columns(
    pl.when(pl.Series(np.random.rand(100) > 0.5)).then(None).otherwise(pl.col('IDADE')).alias('IDADE')
)
incomplete.write_parquet(f'{WORKING_DIR}/incomplete.parquet')

!datalus inpaint {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/incomplete.parquet {WORKING_DIR}/inpainted.parquet

## 9. Modificação Contrafactual
Aplicando intervenções (ex: alterando 'SEXO') e regenerando campos compatíveis.

In [ ]:
!datalus counterfactual {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet {WORKING_DIR}/counterfactual.parquet '{"SEXO": "1"}'

## 10. Auditoria Autônoma de Privacidade e Utilidade
Avaliando métricas de DCR e razões de utilidade TSTR/TRTR.

In [ ]:
!datalus audit {WORKING_DIR}/processed.parquet {WORKING_DIR}/synthetic.parquet {WORKING_DIR}/schema_config.json {WORKING_DIR}/audit_report.json --target-column MORTE --mia-mode ci_lite

## 11. Visualizando Resultados da Auditoria


In [ ]:
import json
import matplotlib.pyplot as plt

with open(f'{WORKING_DIR}/audit_report.json', 'r') as f:
    report = json.load(f)

print(json.dumps(report, indent=2))

# Plotar Utilidade (TSTR vs TRTR)
if 'utility' in report and 'target_class_probs' in report['utility']:
    probs = report['utility']['target_class_probs']
    labels = list(probs.keys())
    real_probs = [probs[l]['real'] for l in labels]
    synth_probs = [probs[l]['synthetic'] for l in labels]
    
    x = np.arange(len(labels))
    width = 0.35
    
    fig, ax = plt.subplots()
    ax.bar(x - width/2, real_probs, width, label='Real')
    ax.bar(x + width/2, synth_probs, width, label='Synthetic')
    
    ax.set_ylabel('Probability')
    ax.set_title('Target Class Distribution Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    plt.show()

## 12. Exportando para ONNX Edge
Exportando pesos para inferência local no navegador ou edge.

In [ ]:
!datalus export-onnx {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/artifacts --quantize

## 13. Servindo e Implantação Web
Os artefatos podem ser servidos via FastAPI:
`datalus serve {WORKING_DIR}/artifacts --host 0.0.0.0 --port 8000`

Isso permite que o frontend React WASM realize inferência localmente no navegador do usuário.